<a href="https://colab.research.google.com/github/Joaoplims/NLP-HandsOn/blob/main/HO04/HO04_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#HO04 - CLASSIFICAÇÃO TEXTUAL



### Leitura de dados


In [1]:
import pandas as pd
from datasets import load_dataset
import kagglehub
import os
import glob

def process_url_and_create_df(dataset_item: dict) -> pd.DataFrame | None:
    """
    Loads data based on the source specified in the dataset_item dictionary.
    Supports: 'Github', 'Hugging Face', and 'Kaggle'.
    """
    source = dataset_item.get('source')
    address = dataset_item.get('address')

    try:
        if source == "Github":
            print(f"Downloading directly from GitHub: {address}")
            try:
                return pd.read_csv(address)
            except UnicodeDecodeError:
                print("UTF-8 decoding failed. Retrying with latin-1 encoding...")
                return pd.read_csv(address, encoding='latin-1')

        elif source == "Hugging Face":
            print(f"Loading Hugging Face dataset: {address}")
            ds = load_dataset(address)
            if hasattr(ds, 'keys'):
                split_name = 'train' if 'train' in ds else list(ds.keys())[0]
                return ds[split_name].to_pandas()
            return ds.to_pandas()

        elif source == "Kaggle":
            print(f"Downloading Kaggle dataset: {address}")
            path = kagglehub.dataset_download(address)
            print(f"Path to dataset files: {path}")

            # Search for CSV or JSONL files
            csv_files = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
            jsonl_files = glob.glob(os.path.join(path, "**", "*.jsonl"), recursive=True)

            if csv_files:
                print(f"Found CSV: {os.path.basename(csv_files[0])}")
                return pd.read_csv(csv_files[0])
            elif jsonl_files:
                print(f"Found JSONL: {os.path.basename(jsonl_files[0])}")
                return pd.read_json(jsonl_files[0], lines=True)
            else:
                all_files = glob.glob(os.path.join(path, "**", "*"), recursive=True)
                print(f"No supported files found. Files available: {[os.path.basename(f) for f in all_files if os.path.isfile(f)]}")
                return None

        else:
            print(f"Unsupported source: {source}")
            return None

    except Exception as e:
        print(f"Error loading from {source} ({address}): {e}")
        return None

In [2]:
datasetSource = {"source": "Kaggle", "address": "crowdflower/twitter-airline-sentiment"}
airlinesSent_df = process_url_and_create_df(datasetSource)
if airlinesSent_df is not None:
    print(f"Successfully loaded dataset from {datasetSource['source']}:")
    display(airlinesSent_df.head())

Using Colab cache for faster access to the 'twitter-airline-sentiment' dataset.
Path to dataset files: /kaggle/input/twitter-airline-sentiment
Found CSV: Tweets.csv
Successfully loaded dataset from Kaggle:


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


### Text Processing


In [3]:
# Install necessary libraries
!pip install unidecode

# Download NLTK data
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')
# Removed: nltk.download('rslp') as it's for Portuguese
# Download Scikit-Learn
!pip install -U scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 12.1 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 20.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [3]:
import re
from unidecode import unidecode
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Initialize lemmatizer and stemmer
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def normalize_text(text):
    # Remove Twitter mentions (e.g., @username)
    text = re.sub(r'@\w+', '', text)

    # 1. Remove accents and special characters
    text = unidecode(text) # Remove accents
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove special characters, keep letters and spaces
    text = text.lower() # Convert to lowercase

    # 2. Tokenization
    tokens = word_tokenize(text)

    # 3. Lemmatization (using English WordNetLemmatizer)
    lemmas = [lemmatizer.lemmatize(word) for word in tokens]

    # 4. Stemming (using English PorterStemmer)
    stemmed_tokens = [stemmer.stem(word) for word in lemmas] # Apply stemming after lemmatization

    return stemmed_tokens

print("Text normalization function 'normalize_text' created successfully.")

Text normalization function 'normalize_text' created successfully.


In [5]:
# Applying the normalization pipeline to the 'text' column
# The result is saved as a list of tokenized/normalized sentences (an array structure)
if 'text' in airlinesSent_df.columns:
    normalized_data = airlinesSent_df['text'].apply(normalize_text).tolist()

    print(f"Normalization complete. Total records: {len(normalized_data)}")
    print("Example of the first normalized record:")
    print(normalized_data [:5])
else:
    print("Error: Column 'text' not found in the DataFrame.")

Normalization complete. Total records: 14640
Example of the first normalized record:
[['what', 'said'], ['plu', 'youv', 'ad', 'commerci', 'to', 'the', 'experi', 'tacki'], ['i', 'didnt', 'today', 'must', 'mean', 'i', 'need', 'to', 'take', 'anoth', 'trip'], ['it', 'realli', 'aggress', 'to', 'blast', 'obnoxi', 'entertain', 'in', 'your', 'guest', 'face', 'amp', 'they', 'have', 'littl', 'recours'], ['and', 'it', 'a', 'realli', 'big', 'bad', 'thing', 'about', 'it']]


### Representação Vetorial

#### TF IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# O TfidfVectorizer espera uma lista de strings. Convertendo cada lista de tokens da coluna Text em uma string unica
normalized_strings = [" ".join(tokens) for tokens in normalized_data]

# Gerar a Matriz TF-IDF usando TfidfVectorizer
print("Gerando a Matriz TF-IDF usando TfidfVectorizer...")
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix_sklearn = tfidf_vectorizer.fit_transform(normalized_strings)

print(f"Formato da Matriz TF-IDF (Sklearn): {tfidf_matrix_sklearn.shape}")
print(f"Exemplo de feature names: {tfidf_vectorizer.get_feature_names_out()[:5]}")

print("\nExemplo dos 10 primeiros termos não-zero da primeira linha (Sklearn):")
first_row_sklearn = tfidf_matrix_sklearn[0].tocoo()
feature_names_sklearn = tfidf_vectorizer.get_feature_names_out()

non_zero_terms_count = 0
for r, c, v in zip(first_row_sklearn.row, first_row_sklearn.col, first_row_sklearn.data):
    if v != 0 and non_zero_terms_count < 10:
        print(f"  Termo: {feature_names_sklearn[c]} | Peso: {v:.4f}")
        non_zero_terms_count += 1
    if non_zero_terms_count >= 10:
        break

Gerando a Matriz TF-IDF usando TfidfVectorizer...
Formato da Matriz TF-IDF (Sklearn): (14640, 10968)
Exemplo de feature names: ['aa' 'aaaand' 'aaadvantag' 'aaalwaysl' 'aaba']

Exemplo dos 10 primeiros termos não-zero da primeira linha (Sklearn):
  Termo: what | Peso: 0.5832
  Termo: said | Peso: 0.8124


In [ ]:
# Testando...
print(tfidf_matrix_sklearn)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 208968 stored elements and shape (14640, 10968)>
  Coords	Values
  (0, 10585)	0.5831694027806291
  (0, 8509)	0.8123505694344557
  (1, 7726)	0.36183888009229964
  (1, 10929)	0.3627748886310162
  (1, 89)	0.3742660115884859
  (1, 1641)	0.4288895668321668
  (1, 9797)	0.10249536614343152
  (1, 9622)	0.11792634323711963
  (1, 2840)	0.29180644394895366
  (1, 9471)	0.5512556338253416
  (2, 9797)	0.12194888913685677
  (2, 2257)	0.34719102942422997
  (2, 9801)	0.30269061639273187
  (2, 6770)	0.4643635422544978
  (2, 6423)	0.3993724881872256
  (2, 6844)	0.27142732907995515
  (2, 9481)	0.3101550184683647
  (2, 384)	0.3315937393910296
  (2, 9934)	0.34352587402043194
  (3, 9797)	0.07254323421120361
  (3, 5590)	0.11421528518728198
  (3, 8112)	0.19332291614708413
  (3, 161)	0.3628411682475392
  (3, 903)	0.3407830127312218
  (3, 7151)	0.3741806194034993
  :	:
  (14638, 7609)	0.21210790094846427
  (14638, 1240)	0.16799106365479355
  (14638, 1

#### Word2Vec

In [ ]:
!pip install --upgrade gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 25.3 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec
import logging

# 1. Preparar os dados (Word2Vec espera uma lista de listas de tokens)
print("Preparando tokens para o Word2Vec...")
normalized_strings_W2V = normalized_data

# 2. Treinar o modelo Word2Vec
print("Treinando o modelo Word2Vec (isso pode levar alguns instantes)...")
w2v_model = Word2Vec(sentences= normalized_strings_W2V,
                     vector_size=100,
                     window=5,
                     min_count=2,
                     workers=4)

print("Modelo Word2Vec treinado com sucesso!")

Preparando tokens para o Word2Vec...
Treinando o modelo Word2Vec (isso pode levar alguns instantes)...
Modelo Word2Vec treinado com sucesso!


In [ ]:
# 3. Demonstrando o Word2Vec: Encontrando palavras similares
# Nota: Usamos 'commerci' porque o Stemmer reduziu 'commercials' a esse radical
word_to_test = "commerci"

if word_to_test in w2v_model.wv:
    print(f"Palavras mais similares a '{word_to_test}':")
    similar_words = w2v_model.wv.most_similar(word_to_test, topn=5)
    for word, score in similar_words:
        print(f"  {word}: {score:.4f}")

    # Exemplo de vetor para uma palavra (movido para dentro do IF para evitar erro)
    print(f"\nFormato do vetor da palavra '{word_to_test}': {w2v_model.wv[word_to_test].shape}")
else:
    print(f"A palavra '{word_to_test}' não foi encontrada no vocabulário do modelo.")
    print("Dica: Verifique se a palavra não foi alterada pelo Stemmer/Lemmatizer ou se ela aparece menos que o 'min_count'.")

Palavras mais similares a 'commerci':
  perform: 0.9969
  rais: 0.9968
  situat: 0.9968
  program: 0.9966
  flyer: 0.9965

Formato do vetor da palavra 'commerci': (100,)


### Naive Bayes Classifier

#### Versão TF-IDF

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# 1. Definindo as variáveis dependentes (X) e independentes (y)
# X é a nossa matriz TF-IDF e y é a coluna de sentimento
X = tfidf_matrix_sklearn
y = airlinesSent_df['airline_sentiment']

# 2. Divisão de treino e teste (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Instanciar o modelo MultinomialNB
nb_model = MultinomialNB()

# 4. Treinar o modelo
nb_model.fit(X_train, y_train)

# 5. Avaliação básica
y_pred = nb_model.predict(X_test)
print(f"Acurácia do modelo: {accuracy_score(y_test, y_pred):.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred))

Acurácia do modelo: 0.6786

Relatório de Classificação:
              precision    recall  f1-score   support

    negative       0.67      1.00      0.80      1835
     neutral       0.78      0.13      0.22       620
    positive       0.88      0.16      0.27       473

    accuracy                           0.68      2928
   macro avg       0.78      0.43      0.43      2928
weighted avg       0.73      0.68      0.59      2928



#### Versao Word2Vec

In [ ]:
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

def get_document_vector(tokens, model):
    # Filtra palavras que existem no vocabulário do Word2Vec
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]

    # Se o tweet não tiver nenhuma palavra conhecida, retorna um vetor de zeros
    if not valid_vectors:
        return np.zeros(model.vector_size)

    # Retorna a média aritmética dos vetores
    return np.mean(valid_vectors, axis=0)

# 1. Converter a lista de tokens (normalized_data) em um array de vetores (X_w2v)
print("Convertendo documentos em vetores médios...")
X_w2v = np.array([get_document_vector(tokens, w2v_model) for tokens in normalized_data])

# 2. Definir o alvo (y)
y = airlinesSent_df['airline_sentiment']

# 3. Divisão de treino e teste
X_train_w2v, X_test_w2v, y_train_w2v, y_test_w2v = train_test_split(
    X_w2v, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Treinar o modelo GaussianNB (adequado para vetores contínuos com valores negativos)
gnb_model = GaussianNB()
gnb_model.fit(X_train_w2v, y_train_w2v)

# 5. Avaliação
y_pred_w2v = gnb_model.predict(X_test_w2v)

print(f"Formato da matriz X_w2v: {X_w2v.shape}")
print(f"Acurácia (Word2Vec + GaussianNB): {accuracy_score(y_test_w2v, y_pred_w2v):.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test_w2v, y_pred_w2v))

Convertendo documentos em vetores médios...
Formato da matriz X_w2v: (14640, 100)
Acurácia (Word2Vec + GaussianNB): 0.6270

Relatório de Classificação:
              precision    recall  f1-score   support

    negative       0.73      0.77      0.75      1835
     neutral       0.43      0.37      0.40       620
    positive       0.42      0.41      0.42       473

    accuracy                           0.63      2928
   macro avg       0.53      0.52      0.52      2928
weighted avg       0.62      0.63      0.62      2928



### Decision Tree Classifier

#### Versão TF-IDF

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Instanciar o modelo de Árvore de Decisão
dt_model = DecisionTreeClassifier(random_state=42)

# 2. Treinar o modelo usando os mesmos dados de treino do TF-IDF (X_train, y_train)
dt_model.fit(X_train, y_train)

# 3. Realizar as previsões no conjunto de teste
y_pred_dt = dt_model.predict(X_test)

# 4. Exibir resultados
print(f"Acurácia do Decision Tree (TF-IDF): {accuracy_score(y_test, y_pred_dt):.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred_dt))

Acurácia do Decision Tree (TF-IDF): 0.7001

Relatório de Classificação:
              precision    recall  f1-score   support

    negative       0.79      0.82      0.81      1835
     neutral       0.49      0.49      0.49       620
    positive       0.59      0.53      0.55       473

    accuracy                           0.70      2928
   macro avg       0.62      0.61      0.62      2928
weighted avg       0.70      0.70      0.70      2928



#### Versão Word2Vec

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Instanciar o modelo de Árvore de Decisão
dt_model_w2v = DecisionTreeClassifier(random_state=42)

# 2. Treinar o modelo usando os dados de treino do Word2Vec (X_train_w2v, y_train_w2v)
dt_model_w2v.fit(X_train_w2v, y_train_w2v)

# 3. Realizar as previsões no conjunto de teste
y_pred_dt_w2v = dt_model_w2v.predict(X_test_w2v)

# 4. Exibir os resultados
print(f"Acurácia do Decision Tree (Word2Vec): {accuracy_score(y_test_w2v, y_pred_dt_w2v):.4f}")
print("\nRelatório de Classificação (Word2Vec):")
print(classification_report(y_test_w2v, y_pred_dt_w2v))

Acurácia do Decision Tree (Word2Vec): 0.5997

Relatório de Classificação (Word2Vec):
              precision    recall  f1-score   support

    negative       0.74      0.72      0.73      1835
     neutral       0.39      0.41      0.40       620
    positive       0.37      0.37      0.37       473

    accuracy                           0.60      2928
   macro avg       0.50      0.50      0.50      2928
weighted avg       0.60      0.60      0.60      2928



### Support Vector Machines (SVM)

#### Versão TF-IDF

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# 1. Instanciar o modelo SVM
# Usamos o kernel 'linear' que geralmente funciona bem para classificação de texto com TF-IDF
svm_model = SVC(kernel='linear', random_state=42)

# 2. Treinar o modelo utilizando os conjuntos de treino do TF-IDF
print("Treinando o modelo SVM (isso pode levar alguns segundos)... ")
svm_model.fit(X_train, y_train)

# 3. Predição e Avaliação
y_pred_svm = svm_model.predict(X_test)

print(f"Acurácia do modelo SVM (TF-IDF): {accuracy_score(y_test, y_pred_svm):.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred_svm))

Treinando o modelo SVM (isso pode levar alguns segundos)... 
Acurácia do modelo SVM (TF-IDF): 0.7951

Relatório de Classificação:
              precision    recall  f1-score   support

    negative       0.83      0.92      0.88      1835
     neutral       0.66      0.58      0.62       620
    positive       0.79      0.59      0.68       473

    accuracy                           0.80      2928
   macro avg       0.76      0.70      0.72      2928
weighted avg       0.79      0.80      0.79      2928



#### Versão Word2Vec

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# 1. Instanciar o modelo SVM
# Mantemos o kernel 'linear' para consistência na comparação
svm_model_w2v = SVC(kernel='linear', random_state=42)

# 2. Treinar o modelo utilizando os conjuntos de treino do Word2Vec
print("Treinando o modelo SVM com Word2Vec... ")
svm_model_w2v.fit(X_train_w2v, y_train_w2v)

# 3. Predição e Avaliação
y_pred_svm_w2v = svm_model_w2v.predict(X_test_w2v)

print(f"Acurácia do modelo SVM (Word2Vec): {accuracy_score(y_test_w2v, y_pred_svm_w2v):.4f}")
print("\nRelatório de Classificação (Word2Vec):")
print(classification_report(y_test_w2v, y_pred_svm_w2v))

Treinando o modelo SVM com Word2Vec... 
Acurácia do modelo SVM (Word2Vec): 0.6957

Relatório de Classificação (Word2Vec):
              precision    recall  f1-score   support

    negative       0.68      0.99      0.81      1835
     neutral       0.71      0.12      0.20       620
    positive       0.85      0.33      0.48       473

    accuracy                           0.70      2928
   macro avg       0.75      0.48      0.49      2928
weighted avg       0.72      0.70      0.63      2928



### Logistic Regression Classifier
#### Versão TF-IDF

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Instanciar o modelo de Regressão Logística
# Aumentamos o max_iter para garantir a convergência
lr_model = LogisticRegression(max_iter=1000, random_state=42)

# 2. Treinar o modelo utilizando os conjuntos de treino do TF-IDF
print("Treinando o modelo Logistic Regression com TF-IDF...")
lr_model.fit(X_train, y_train)

# 3. Predição e Avaliação
y_pred_lr = lr_model.predict(X_test)

print(f"Acurácia do Logistic Regression (TF-IDF): {accuracy_score(y_test, y_pred_lr):.4f}")
print("\nRelatório de Classificação:")
print(classification_report(y_test, y_pred_lr))

Treinando o modelo Logistic Regression com TF-IDF...
Acurácia do Logistic Regression (TF-IDF): 0.7978

Relatório de Classificação:
              precision    recall  f1-score   support

    negative       0.83      0.94      0.88      1835
     neutral       0.67      0.57      0.62       620
    positive       0.82      0.54      0.65       473

    accuracy                           0.80      2928
   macro avg       0.77      0.68      0.72      2928
weighted avg       0.79      0.80      0.79      2928



#### Versão Word2Vec

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Instanciar o modelo de Regressão Logística
lr_model_w2v = LogisticRegression(max_iter=1000, random_state=42)

# 2. Treinar o modelo utilizando os conjuntos de treino do Word2Vec
print("Treinando o modelo Logistic Regression com Word2Vec...")
lr_model_w2v.fit(X_train_w2v, y_train_w2v)

# 3. Predição e Avaliação
y_pred_lr_w2v = lr_model_w2v.predict(X_test_w2v)

print(f"Acurácia do Logistic Regression (Word2Vec): {accuracy_score(y_test_w2v, y_pred_lr_w2v):.4f}")
print("\nRelatório de Classificação (Word2Vec):")
print(classification_report(y_test_w2v, y_pred_lr_w2v))

Treinando o modelo Logistic Regression com Word2Vec...
Acurácia do Logistic Regression (Word2Vec): 0.7258

Relatório de Classificação (Word2Vec):
              precision    recall  f1-score   support

    negative       0.74      0.95      0.83      1835
     neutral       0.62      0.34      0.44       620
    positive       0.75      0.38      0.50       473

    accuracy                           0.73      2928
   macro avg       0.70      0.55      0.59      2928
weighted avg       0.72      0.73      0.69      2928



### XGBoost Classifier

In [ ]:
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# O XGBoost exige que os rótulos (y) sejam numéricos
le = LabelEncoder()
y_train_num = le.fit_transform(y_train)
y_test_num = le.transform(y_test)

# 1. Instanciar e treinar o modelo para TF-IDF
print("Treinando XGBoost com TF-IDF...")
xgb_model_tfidf = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
xgb_model_tfidf.fit(X_train, y_train_num)

# 2. Predição e Avaliação
y_pred_xgb_tfidf = xgb_model_tfidf.predict(X_test)

print(f"Acurácia do XGBoost (TF-IDF): {accuracy_score(y_test_num, y_pred_xgb_tfidf):.4f}")
print("\nRelatório de Classificação TF-IDF:")
print(classification_report(y_test_num, y_pred_xgb_tfidf, target_names=le.classes_))

Treinando XGBoost com TF-IDF...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:03:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Acurácia do XGBoost (TF-IDF): 0.7862

Relatório de Classificação TF-IDF:
              precision    recall  f1-score   support

    negative       0.82      0.92      0.86      1835
     neutral       0.65      0.55      0.60       620
    positive       0.80      0.58      0.67       473

    accuracy                           0.79      2928
   macro avg       0.76      0.68      0.71      2928
weighted avg       0.78      0.79      0.78      2928



In [ ]:
# 3. Instanciar e treinar o modelo para Word2Vec
print("Treinando XGBoost com Word2Vec...")
xgb_model_w2v = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
xgb_model_w2v.fit(X_train_w2v, y_train_num)

# 4. Predição e Avaliação
y_pred_xgb_w2v = xgb_model_w2v.predict(X_test_w2v)

print(f"Acurácia do XGBoost (Word2Vec): {accuracy_score(y_test_num, y_pred_xgb_w2v):.4f}")
print("\nRelatório de Classificação Word2Vec:")
print(classification_report(y_test_num, y_pred_xgb_w2v, target_names=le.classes_))

Treinando XGBoost com Word2Vec...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:04:11] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Acurácia do XGBoost (Word2Vec): 0.7196

Relatório de Classificação Word2Vec:
              precision    recall  f1-score   support

    negative       0.76      0.90      0.82      1835
     neutral       0.56      0.42      0.48       620
    positive       0.69      0.42      0.52       473

    accuracy                           0.72      2928
   macro avg       0.67      0.58      0.61      2928
weighted avg       0.71      0.72      0.70      2928



### Random Forest Classifier
O Random Forest cria diversas árvores de decisão e combina seus resultados para melhorar a precisão e controlar o sobreajuste.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Instanciar e treinar o modelo para TF-IDF
print("Treinando Random Forest com TF-IDF...")
rf_model_tfidf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model_tfidf.fit(X_train, y_train)

# 2. Predição e Avaliação
y_pred_rf_tfidf = rf_model_tfidf.predict(X_test)

print(f"Acurácia do Random Forest (TF-IDF): {accuracy_score(y_test, y_pred_rf_tfidf):.4f}")
print("\nRelatório de Classificação TF-IDF:")
print(classification_report(y_test, y_pred_rf_tfidf))

Treinando Random Forest com TF-IDF...
Acurácia do Random Forest (TF-IDF): 0.7592

Relatório de Classificação TF-IDF:
              precision    recall  f1-score   support

    negative       0.76      0.97      0.85      1835
     neutral       0.70      0.38      0.49       620
    positive       0.86      0.45      0.59       473

    accuracy                           0.76      2928
   macro avg       0.77      0.60      0.64      2928
weighted avg       0.76      0.76      0.73      2928



In [ ]:
# 3. Instanciar e treinar o modelo para Word2Vec
print("Treinando Random Forest com Word2Vec...")
rf_model_w2v = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model_w2v.fit(X_train_w2v, y_train_w2v)

# 4. Predição e Avaliação
y_pred_rf_w2v = rf_model_w2v.predict(X_test_w2v)

print(f"Acurácia do Random Forest (Word2Vec): {accuracy_score(y_test_w2v, y_pred_rf_w2v):.4f}")
print("\nRelatório de Classificação Word2Vec:")
print(classification_report(y_test_w2v, y_pred_rf_w2v))

Treinando Random Forest com Word2Vec...
Acurácia do Random Forest (Word2Vec): 0.7114

Relatório de Classificação Word2Vec:
              precision    recall  f1-score   support

    negative       0.73      0.93      0.82      1835
     neutral       0.58      0.35      0.44       620
    positive       0.73      0.33      0.45       473

    accuracy                           0.71      2928
   macro avg       0.68      0.54      0.57      2928
weighted avg       0.70      0.71      0.68      2928



## Relatório Final

### Comparativo Final de Modelos

| Algoritmo | Vetorização | Acurácia | F1-Score (Weighted) |
| :--- | :--- | :---: | :---: |
| **Naive Bayes** | TF-IDF | 67.86% | 0.59 |
| **Naive Bayes (Gaussian)** | Word2Vec | 62.70% | 0.62 |
| **Decision Tree** | TF-IDF | 70.01% | 0.70 |
| **Decision Tree** | Word2Vec | 59.97% | 0.60 |
| **SVM (Linear)** | TF-IDF | 79.51% | 0.79 |
| **SVM (Linear)** | Word2Vec | 69.57% | 0.63 |
| **Logistic Regression** | TF-IDF | 79.78% | 0.79 |
| **Logistic Regression** | Word2Vec | 72.58% | 0.69 |
| **XGBoost** | TF-IDF | 78.62% | 0.78 |
| **XGBoost** | Word2Vec | 71.96% | 0.70 |
| **Random Forest** | TF-IDF | 75.92% | 0.73 |
| **Random Forest** | Word2Vec | 71.14% | 0.68 |

#### Conclusões:
1. **Melhor Performance:** O modelo de **Regressão Logística com TF-IDF** obteve a maior acurácia (**79.78%**), seguido de perto pelo SVM.
2. **TF-IDF vs Word2Vec:** Para este conjunto de dados de tweets, o **TF-IDF** superou consistentemente o Word2Vec em quase todos os algoritmos. Isso sugere que a presença de palavras-chave específicas é um indicador mais forte de sentimento do que a semântica vetorial média neste contexto.
3. **Desbalanceamento:** Todos os modelos apresentaram dificuldades em classificar as classes minoritárias ('neutral' e 'positive'), com o 'recall' sendo significativamente menor do que o da classe 'negative'.

### Entendendo as Métricas de Avaliação

Ao analisar um relatório de classificação do Scikit-Learn, utilizamos diversas métricas para entender a performance do modelo além da simples acurácia:

1.  **Precision (Precisão):** Indica a proporção de predições positivas que foram realmente corretas. *Pergunta: De todos os tweets que o modelo disse ser 'Negativo', quantos eram realmente negativos?*
2.  **Recall (Revocação/Sensibilidade):** Indica a proporção de casos positivos reais que foram identificados corretamente. *Pergunta: De todos os tweets que eram 'Negativos' na vida real, quantos o modelo conseguiu encontrar?*
3.  **F1-Score:** É a média harmônica entre Precision e Recall. É muito útil quando temos classes desbalanceadas, pois busca um equilíbrio entre as duas métricas.
4.  **Support (Suporte):** O número real de ocorrências de cada classe no conjunto de teste. Ajuda a identificar se os dados estão desbalanceados.
5.  **Accuracy (Acurácia):** A porcentagem total de acertos do modelo (predições corretas / total de casos).
6.  **Macro Avg (Média Macro):** Calcula a métrica para cada classe e tira a média aritmética simples. Trata todas as classes com o mesmo peso, independentemente de quantas amostras cada uma tem.
7.  **Weighted Avg (Média Ponderada):** Calcula a média das métricas levando em conta o suporte (peso) de cada classe. Se uma classe tem muito mais exemplos (como o 'negative' neste dataset), ela influenciará mais o resultado final desta média.

## EXTRAS